In [ ]:
!unzip /content/screenshots.zip -d /content/screenshots

Archive:  /content/screenshots.zip
   creating: /content/screenshots/screenshots/
  inflating: /content/screenshots/screenshots/60061.jpg  
  inflating: /content/screenshots/screenshots/58419.jpg  
  inflating: /content/screenshots/screenshots/42214.jpg  
  inflating: /content/screenshots/screenshots/47883.jpg  
  inflating: /content/screenshots/screenshots/70502.jpg  
  inflating: /content/screenshots/screenshots/10798.jpg  
  inflating: /content/screenshots/screenshots/66870.jpg  
  inflating: /content/screenshots/screenshots/14247.jpg  
  inflating: /content/screenshots/screenshots/11150.jpg  
  inflating: /content/screenshots/screenshots/28535.jpg  
  inflating: /content/screenshots/screenshots/14554.jpg  
  inflating: /content/screenshots/screenshots/8869.jpg  
  inflating: /content/screenshots/screenshots/58270.jpg  
  inflating: /content/screenshots/screenshots/26425.jpg  
  inflating: /content/screenshots/screenshots/1996.jpg  
  inflating: /content/screenshots/screenshots/6666

In [ ]:
# ============================================
# CELL 1 — Install dependencies (KEEP JSON REPAIR)
# ============================================
!pip install -q torch torchvision pillow transformers faiss-cpu
!pip install -q "langchain>=0.2.0" "langchain-community>=0.2.0"
!pip install -q json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
# ============================================
# CELL 2 — Imports
# ============================================
import os, csv, json, re, traceback
import numpy as np
from PIL import Image, UnidentifiedImageError
import torch

from transformers import (
    CLIPProcessor,
    CLIPModel,
    AutoProcessor,
    AutoModelForVision2Seq,
)

import faiss

from langchain_community.vectorstores import FAISS
from langchain_community.docstore import InMemoryDocstore
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings

from json_repair import repair_json

In [ ]:
# ============================================
# CELL 3 — Paths (adjust if needed)
# ============================================
screenshot_dir   = "/content/screenshots/screenshots"
topic_csv_path   = "/content/design_topics.csv"
save_dir         = "ss_faiss_langchain"

# where description .txt files will be stored
desc_txt_dir     = "/content/drive/MyDrive/SS_descriptions_txt_final"

# where generated per-screen test case CSVs will be stored
tc_csv_dir       = "/content/drive/MyDrive/SS_testcases_csv_final"

os.makedirs(desc_txt_dir, exist_ok=True)
os.makedirs(tc_csv_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
# ============================================
# CELL 4 — OPTIONAL: Mount Google Drive (so outputs persist)
# If you don't want Drive, you can skip this cell.
# ============================================
# Choose a Drive output root (edit if you want)
DRIVE_ROOT = "/content/drive/MyDrive/SS_PIPELINE_200_final1"

DESC_DRIVE_DIR = os.path.join(DRIVE_ROOT, "descriptions_txt")
TC_DRIVE_DIR   = os.path.join(DRIVE_ROOT, "testcases_csv")

os.makedirs(DESC_DRIVE_DIR, exist_ok=True)
os.makedirs(TC_DRIVE_DIR, exist_ok=True)

print("Drive output:")
print("  DESC_DRIVE_DIR:", DESC_DRIVE_DIR)
print("  TC_DRIVE_DIR:  ", TC_DRIVE_DIR)


Drive output:
  DESC_DRIVE_DIR: /content/drive/MyDrive/SS_PIPELINE_200_final1/descriptions_txt
  TC_DRIVE_DIR:   /content/drive/MyDrive/SS_PIPELINE_200_final1/testcases_csv


In [ ]:
# ============================================
# CELL 5 — Load models (CLIP + Qwen2.5-VL)
# ============================================
clip_name = "laion/CLIP-ViT-L-14-laion2B-s32B-b82K"
clip_model = CLIPModel.from_pretrained(clip_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_name)

MAX_PROMPT_CHARS = 2200
MAX_NEW_TOKENS   = 700

vlm_name = "Qwen/Qwen2.5-VL-7B-Instruct"
vlm_processor = AutoProcessor.from_pretrained(vlm_name, trust_remote_code=True)
vlm_model = AutoModelForVision2Seq.from_pretrained(
    vlm_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
).to(device)

print("Loaded CLIP + Qwen2.5-VL-7B-Instruct.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Loaded CLIP + Qwen2.5-VL-7B-Instruct.


In [ ]:
# ============================================
# CELL 6 — Safe image loading
# ============================================
def load_image_safe(path):
    try:
        img = Image.open(path)
    except UnidentifiedImageError:
        print(f"[SKIP] Unreadable image: {path}")
        return None
    except Exception as e:
        print(f"[SKIP] Error opening {path}: {e}")
        return None

    if img.mode == "P":
        img = img.convert("RGBA")
    if img.mode in ("RGBA", "LA"):
        img = img.convert("RGB")
    else:
        img = img.convert("RGB")
    return img


def collect_valid_images(root):
    valid, bad = [], []
    for fname in sorted(os.listdir(root)):
        if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
            continue
        path = os.path.join(root, fname)
        img = load_image_safe(path)
        if img is None:
            bad.append(path)
        else:
            valid.append(path)
    return valid, bad


def get_id_from_path(path):
    return os.path.splitext(os.path.basename(path))[0]


In [ ]:
# ============================================
# CELL 7 — CLIP embedding helpers
# ============================================
@torch.no_grad()
def embed_pil_image(img: Image.Image):
    inputs = clip_processor(images=img, return_tensors="pt").to(device)
    emb = clip_model.get_image_features(**inputs)
    emb = emb / emb.norm(p=2, dim=-1, keepdim=True)
    return emb.squeeze().detach().cpu().numpy().astype("float32")


@torch.no_grad()
def embed_image(path):
    img = load_image_safe(path)
    if img is None:
        return None
    return embed_pil_image(img)


In [ ]:
# ============================================
# CELL 8 — Qwen output cleanup + VLM generation (image + text)
# ============================================
def _strip_qwen_chat_header(text: str) -> str:
    marker = "assistant\n"
    last_idx = text.rfind(marker)
    if last_idx != -1:
        return text[last_idx + len(marker):].strip()
    return text.strip()


@torch.no_grad()
def generate_with_vlm(img: Image.Image, prompt_text: str, max_new_tokens=MAX_NEW_TOKENS):
    prompt_text = prompt_text[:MAX_PROMPT_CHARS]

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    chat_text = vlm_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = vlm_processor(
        text=[chat_text],
        images=[img],
        return_tensors="pt",
    ).to(device)

    generated_ids = vlm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=3,
        repetition_penalty=1.08,
    )

    raw = vlm_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )[0]

    return _strip_qwen_chat_header(raw)


In [ ]:
# ============================================
# CELL 9 — TEXT-ONLY generation (Stage 2 uses this)
# ============================================
@torch.no_grad()
def generate_text_only(prompt_text: str, max_new_tokens=900):
    prompt_text = prompt_text[:MAX_PROMPT_CHARS]

    messages = [
        {"role": "user", "content": [{"type": "text", "text": prompt_text}]}
    ]

    chat_text = vlm_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = vlm_processor(
        text=[chat_text],
        return_tensors="pt",
    ).to(device)

    generated_ids = vlm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=3,
        repetition_penalty=1.08,
    )

    raw = vlm_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return _strip_qwen_chat_header(raw)


In [ ]:
# ============================================
# CELL 10 — Crop status bar (unchanged)
# ============================================
def crop_status_bar(img: Image.Image, top_ratio=0.12):
    w, h = img.size
    top_h = max(40, int(h * top_ratio))
    return img.crop((0, 0, w, top_h))


In [ ]:
# ============================================
# CELL 11 — Prompts (UNCHANGED)
# ============================================
STATUSBAR_PROMPT = """
You are a senior mobile UI analyst.

Focus ONLY on the STATUS BAR (the thin strip at the very top of the screen).
Ignore everything below it.

Your task is to report what is visually present, with confidence awareness.

Rules:
- If a symbol is clearly recognizable by its visual shape or logo, name it.
- If a symbol looks like a known system icon but is not fully certain, write:
  "likely <icon name> (low confidence)".
- If a symbol cannot be reliably identified, write:
  "unknown icon" with a brief shape description.
- Do NOT assume icons just because they are common.
- Do NOT infer icons from context.
- Time should be reported if readable.

Scan strictly from LEFT → RIGHT.

Return EXACTLY two sections:

1) Left-side notification/app icons:
   - List icons in order.
   - Use app names only when logos are unmistakable.

2) Right-side system/status icons + time:
   - List icons in order.
   - Use confidence labels where needed.
   - End with time if readable.

Be literal, visual, and conservative.
"""

SCREENSHOT_PROMPT = """
You are a senior mobile UI and UX analyst.

You are looking at ONE FINAL IMPLEMENTED MOBILE APP SCREENSHOT (a fully designed UI).
Describe the screen in detail for a developer who cannot see the image.

ABSOLUTE RULES:
- Always talk in terms of UI components (status bar, header/app bar, title, subtitle, image, list row, toggle, chevron, button, footer, bottom nav).
- If any text is readable (titles, labels, button text), include it verbatim.
- Do NOT skip the bottom area: capture CTA buttons and any footer/bottom nav if present.

OUTPUT STRUCTURE (use these headings exactly):
A) Screen purpose and type:
B) Layout top → bottom:
C) Component inventory (grouped):
D) Primary actions + navigation:
Write 14–22 sentences total (compact, but complete).
End with a bullet recap:
- Screen type/purpose
- Key components
- Primary CTA(s)
- Navigation/footer
"""


In [ ]:
# ============================================
# CELL 12 — Caches
# ============================================
_screenshot_desc_cache = {}
_statusbar_cache = {}


In [ ]:
# ============================================
# CELL 13 — Load topics CSV: screen_id -> topic
# ============================================
id_to_topic = {}
with open(topic_csv_path, newline="") as f:
    reader = csv.reader(f)
    header = next(reader, None)
    for row in reader:
        if not row:
            continue
        sid   = row[0].strip()
        topic = row[1].strip() if len(row) > 1 else "unknown"
        id_to_topic[sid] = topic

print("Total IDs with topics:", len(id_to_topic))


Total IDs with topics: 1460


In [ ]:
# ============================================
# CELL 14 — Collect screenshots + mapping
# ============================================
valid_screenshots, bad_screenshots = collect_valid_images(screenshot_dir)
print(f"Valid screenshots:      {len(valid_screenshots)}")
print(f"Unreadable screenshots: {len(bad_screenshots)}")

screenshot_by_id = {}
for p in valid_screenshots:
    pid = get_id_from_path(p)
    screenshot_by_id[pid] = p

print("Example screenshot_by_id keys:", list(screenshot_by_id.keys())[:5])


Valid screenshots:      1460
Unreadable screenshots: 0
Example screenshot_by_id keys: ['10128', '10180', '10183', '10208', '10214']


In [ ]:
# ============================================
# CELL 15 — Stage 1: Describe screenshot + save .txt (Local + Drive)
# ============================================
def save_description_txt(screen_id: str, merged_description: str, out_dir: str):
    out_path = os.path.join(out_dir, f"{screen_id}.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(merged_description.strip() + "\n")
    return out_path


def describe_screenshot(path, save_txt=True):
    if path in _screenshot_desc_cache:
        return _screenshot_desc_cache[path]

    img = load_image_safe(path)
    if img is None:
        merged = "Screenshot could not be opened."
        _screenshot_desc_cache[path] = merged
        return merged

    full_desc = generate_with_vlm(img, SCREENSHOT_PROMPT, max_new_tokens=MAX_NEW_TOKENS)

    sb_img = crop_status_bar(img, top_ratio=0.12)
    sb_desc = generate_with_vlm(sb_img, STATUSBAR_PROMPT, max_new_tokens=220)
    _statusbar_cache[path] = sb_desc

    merged = (
        full_desc.strip()
        + "\n\n"
        + "STATUS BAR (cropped pass):\n"
        + sb_desc.strip()
    )

    _screenshot_desc_cache[path] = merged

    if save_txt:
        sid = get_id_from_path(path)
        save_description_txt(sid, merged, out_dir=desc_txt_dir)
        # also save to Drive if mounted
        if os.path.exists("/content/drive"):
            save_description_txt(sid, merged, out_dir=DESC_DRIVE_DIR)

    return merged


In [ ]:
# ============================================
# CELL 16 — (Optional) Build + Save + Reload screenshot vector store
# (unchanged workflow; NOT required for the 200-image batch itself)
# ============================================
def build_and_save_screenshot_vectorstore(paths, save_dir=save_dir):
    ss_vectors = []
    ss_meta = []

    for path in paths:
        sid = get_id_from_path(path)
        vec = embed_image(path)
        if vec is None:
            continue

        topic = id_to_topic.get(sid, "unknown")
        meta = {
            "screen_id": sid,
            "topic": topic,
            "path": path,
        }

        ss_vectors.append(vec)
        ss_meta.append(meta)

    if len(ss_vectors) == 0:
        raise RuntimeError("No valid screenshot embeddings were created.")

    ss_vectors = np.stack(ss_vectors).astype("float32")
    d = ss_vectors.shape[1]

    faiss_index = faiss.IndexFlatIP(d)
    docstore    = InMemoryDocstore({})
    id_map      = {}

    ss_vs_build = FAISS(
        embedding_function=None,
        index=faiss_index,
        docstore=docstore,
        index_to_docstore_id=id_map,
    )

    ss_docs = [Document(page_content="screenshot", metadata=m) for m in ss_meta]
    ids = [str(i) for i in range(len(ss_docs))]
    text_embeddings = [(ss_docs[i].page_content, ss_vectors[i]) for i in range(len(ss_docs))]
    metadatas       = [ss_docs[i].metadata for i in range(len(ss_docs))]

    ss_vs_build.add_embeddings(text_embeddings, metadatas, ids)
    ss_vs_build.save_local(save_dir)

    print(f"Saved screenshot FAISS index to: {save_dir}/")
    print("Index ntotal:", ss_vs_build.index.ntotal)

    return ss_vectors, ss_meta


class DummyEmbeddings(Embeddings):
    def embed_documents(self, texts):
        return [[0.0] * 1 for _ in texts]
    def embed_query(self, text):
        return [0.0] * 1


def load_screenshot_vectorstore(save_dir=save_dir):
    ss_vs = FAISS.load_local(
        save_dir,
        embeddings=DummyEmbeddings(),
        allow_dangerous_deserialization=True,
    )
    print("Reloaded screenshot vector store. ntotal:", ss_vs.index.ntotal)

    ss_meta = []
    ss_vectors = []

    for i in range(ss_vs.index.ntotal):
        doc_id = ss_vs.index_to_docstore_id[i]
        doc    = ss_vs.docstore.search(doc_id)
        ss_meta.append(dict(doc.metadata))
        ss_vectors.append(ss_vs.index.reconstruct(i))

    ss_vectors = np.stack(ss_vectors).astype("float32")
    return ss_vs, ss_vectors, ss_meta


# Uncomment if you want to build/reload:
ss_vectors, ss_meta = build_and_save_screenshot_vectorstore(valid_screenshots, save_dir=save_dir)
ss_vs, ss_vectors_reloaded, ss_meta_reloaded = load_screenshot_vectorstore(save_dir=save_dir)


Saved screenshot FAISS index to: ss_faiss_langchain/
Index ntotal: 1460
Reloaded screenshot vector store. ntotal: 1460


In [ ]:
# ============================================
# CELL 17 — Stage 2 prompt (UNCHANGED)
# ============================================
TESTCASE_PROMPT_TEMPLATE = """
You are a senior QA engineer (mobile apps). Generate REALISTIC, high-quality test cases from the given screen description.

Input is a SCREEN DESCRIPTION extracted from a mobile screenshot (components + actions + navigation + buttons ).
Your goal: produce a set of test cases that a senior tester would actually execute.

Hard rules:
- Do NOT invent UI elements that are not mentioned.
- Do NOT reference "the image" or "the screenshot"; reference only what is described.
- Prefer deterministic, step-by-step actions.
- Cover: positive flows, negative flows, boundary/validation, toggles state, Clickable buttons, Check boxes, Select, Input, navigation, accessibility/usability, interruption/resume, and data persistence where applicable.
- If a test requires unknown app behavior, write assumption in Preconditions (short).

Return ONLY valid JSON (no markdown, no commentary).

JSON schema:
{{
  "module": "<short module name>",
  "screen_id": "<screen id>",
  "test_cases": [
    {{
      "test_case_id": "TC_<screen_id>_###",
      "title": "<clear title>",
      "priority": "P0|P1|P2",
      "type": "Functional|Negative|UI|Accessibility|Usability|Regression",
      "preconditions": ["..."],
      "test_data": ["..."],
      "steps": ["Step 1 ...", "Step 2 ..."],
      "expected": ["..."]
    }}
  ]
}}

Screen ID: {screen_id}
Topic: {topic}

SCREEN DESCRIPTION:
\"\"\"
{description}
\"\"\"
"""


In [ ]:
# ============================================
# CELL 18 — JSON extraction + JSON REPAIR (KEEP THIS)
# ============================================
def _extract_json(text: str):
    text = text.strip()
    if text.startswith("{") and text.endswith("}"):
        return text

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start:end+1]
    return None


def parse_testcases_json(raw_text: str):
    js = _extract_json(raw_text)
    if js is None:
        raise ValueError("Could not locate JSON in model output.")

    # Attempt 1: strict loads
    try:
        return json.loads(js)
    except json.JSONDecodeError:
        pass

    # Attempt 2: remove trailing commas
    js2 = re.sub(r",\s*([}\]])", r"\1", js)
    try:
        return json.loads(js2)
    except json.JSONDecodeError:
        pass

    # Attempt 3: basic quote normalization (rare)
    js3 = js2.replace("“", '"').replace("”", '"').replace("’", "'")
    try:
        return json.loads(js3)
    except json.JSONDecodeError:
        pass

    # Attempt 4: json-repair library (most robust)
    repaired = repair_json(js3)
    try:
        return json.loads(repaired)
    except json.JSONDecodeError as e:
        raise ValueError(f"JSON repair failed. Last error: {e}")


In [ ]:
# ============================================
# CELL 19 — Stage 2: generate test cases from ONE description string
# ============================================
def generate_testcases_from_description(screen_id: str, topic: str, description: str, max_new_tokens=1100):
    prompt = TESTCASE_PROMPT_TEMPLATE.format(
        screen_id=screen_id,
        topic=topic,
        description=description
    )
    raw = generate_text_only(prompt, max_new_tokens=max_new_tokens)
    data = parse_testcases_json(raw)

    if "test_cases" not in data or not isinstance(data["test_cases"], list):
        raise ValueError("JSON missing 'test_cases' list.")
    return data


In [ ]:
# ============================================
# CELL 20 — CSV writers (DEFINE BEFORE USE)
# ============================================
CSV_COLUMNS = [
    "test_case_id",
    "title",
    "module",
    "screen_id",
    "topic",
    "priority",
    "type",
    "preconditions",
    "test_data",
    "test_steps",
    "expected_result"
]

def write_testcases_csv(screen_id: str, topic: str, module: str, test_cases: list, out_dir: str):
    out_path = os.path.join(out_dir, f"{screen_id}.csv")
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        w.writeheader()
        for tc in test_cases:
            w.writerow({
                "test_case_id": tc.get("test_case_id", ""),
                "title": tc.get("title", ""),
                "module": module,
                "screen_id": screen_id,
                "topic": topic,
                "priority": tc.get("priority", ""),
                "type": tc.get("type", ""),
                "preconditions": " | ".join(tc.get("preconditions", []) or []),
                "test_data": " | ".join(tc.get("test_data", []) or []),
                "test_steps": " | ".join(tc.get("steps", []) or []),
                "expected_result": " | ".join(tc.get("expected", []) or []),
            })
    return out_path


def append_to_master_csv(rows: list, master_csv_path: str):
    exists = os.path.exists(master_csv_path)
    with open(master_csv_path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        if not exists:
            w.writeheader()
        for r in rows:
            w.writerow(r)


In [ ]:
# ============================================
# CELL 21 — Stage 2: read description .txt → generate test cases → save CSV
# ============================================
def load_description_txt(screen_id: str, in_dir: str):
    p = os.path.join(in_dir, f"{screen_id}.txt")
    if not os.path.exists(p):
        raise FileNotFoundError(f"Description TXT not found: {p}")
    with open(p, "r", encoding="utf-8") as f:
        return f.read().strip()


def stage2_generate_testcases_for_screen(screen_id: str, master_csv_path: str, tc_out_dir: str):
    topic = id_to_topic.get(screen_id, "unknown")

    # load local description (generated in stage1)
    desc = load_description_txt(screen_id, in_dir=desc_txt_dir)

    data = generate_testcases_from_description(screen_id, topic, desc, max_new_tokens=1200)

    module = (data.get("module", "unknown") or "unknown").strip()
    test_cases = data.get("test_cases", []) or []

    per_csv = write_testcases_csv(screen_id, topic, module, test_cases, out_dir=tc_out_dir)

    rows = []
    for tc in test_cases:
        rows.append({
            "test_case_id": tc.get("test_case_id", ""),
            "title": tc.get("title", ""),
            "module": module,
            "screen_id": screen_id,
            "topic": topic,
            "priority": tc.get("priority", ""),
            "type": tc.get("type", ""),
            "preconditions": " | ".join(tc.get("preconditions", []) or []),
            "test_data": " | ".join(tc.get("test_data", []) or []),
            "test_steps": " | ".join(tc.get("steps", []) or []),
            "expected_result": " | ".join(tc.get("expected", []) or []),
        })

    append_to_master_csv(rows, master_csv_path)
    return per_csv, len(test_cases)


In [ ]:
# ============================================
# CELL 22 — NEW: Batch mode (TOP 200 images) end-to-end
# Generates:
#  - 200 description .txt files
#  - 200 per-screen CSV files
#  - 1 master CSV (all testcases appended)
# Saves BOTH locally + Drive (if mounted)
# ============================================
TAKE_TOP_N = 200

# Choose the top N by sorted filename (stable + reproducible for reviewers)
sorted_paths = sorted(valid_screenshots, key=lambda p: os.path.basename(p).lower())
batch_paths = sorted_paths[:TAKE_TOP_N]

print("Batch size:", len(batch_paths))
print("First 5:", [os.path.basename(p) for p in batch_paths[:5]])
print("Last 5: ", [os.path.basename(p) for p in batch_paths[-5:]])

# Master CSVs (local + drive)
master_csv_local = "/content/testcases_all_top200.csv"
master_csv_drive = os.path.join(TC_DRIVE_DIR, "testcases_all_top200.csv") if os.path.exists("/content/drive") else None

# Clear masters if already exist (fresh run)
if os.path.exists(master_csv_local):
    os.remove(master_csv_local)
if master_csv_drive and os.path.exists(master_csv_drive):
    os.remove(master_csv_drive)

# Summary log (for reviewer-proofing)
summary_csv_local = "/content/summary_top200.csv"
summary_csv_drive = os.path.join(DRIVE_ROOT, "summary_top200.csv") if os.path.exists("/content/drive") else None

if os.path.exists(summary_csv_local):
    os.remove(summary_csv_local)
if summary_csv_drive and os.path.exists(summary_csv_drive):
    os.remove(summary_csv_drive)

SUMMARY_COLS = ["screen_id", "filename", "topic", "desc_saved_local", "desc_saved_drive", "tc_csv_local", "tc_csv_drive", "num_testcases", "status", "error"]
def append_summary(row, path):
    exists = os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=SUMMARY_COLS)
        if not exists:
            w.writeheader()
        w.writerow(row)

ok_count, fail_count = 0, 0

for idx, img_path in enumerate(batch_paths, start=1):
    sid = get_id_from_path(img_path)
    fname = os.path.basename(img_path)
    topic = id_to_topic.get(sid, "unknown")

    row = {
        "screen_id": sid,
        "filename": fname,
        "topic": topic,
        "desc_saved_local": "",
        "desc_saved_drive": "",
        "tc_csv_local": "",
        "tc_csv_drive": "",
        "num_testcases": 0,
        "status": "FAIL",
        "error": ""
    }

    try:
        # Stage 1: description (this function already saves txt local + drive)
        _ = describe_screenshot(img_path, save_txt=True)
        desc_local_path = os.path.join(desc_txt_dir, f"{sid}.txt")
        row["desc_saved_local"] = desc_local_path

        if os.path.exists("/content/drive"):
            desc_drive_path = os.path.join(DESC_DRIVE_DIR, f"{sid}.txt")
            row["desc_saved_drive"] = desc_drive_path

        # Stage 2: testcases -> per-screen CSV + master CSV
        per_csv_local, n_tc = stage2_generate_testcases_for_screen(
            sid,
            master_csv_path=master_csv_local,
            tc_out_dir=tc_csv_dir
        )
        row["tc_csv_local"] = per_csv_local
        row["num_testcases"] = n_tc

        # Also write per-screen CSV to Drive (copy)
        if os.path.exists("/content/drive"):
            # regenerate per-screen into drive folder using already-produced JSON? easiest: copy local file
            per_csv_drive = os.path.join(TC_DRIVE_DIR, f"{sid}.csv")
            with open(per_csv_local, "r", encoding="utf-8") as src, open(per_csv_drive, "w", encoding="utf-8") as dst:
                dst.write(src.read())
            row["tc_csv_drive"] = per_csv_drive

            # append to drive master too by copying local master after each success (simple + safe)
            with open(master_csv_local, "r", encoding="utf-8") as src, open(master_csv_drive, "w", encoding="utf-8") as dst:
                dst.write(src.read())

        row["status"] = "OK"
        ok_count += 1

    except Exception as e:
        fail_count += 1
        row["error"] = str(e).replace("\n", " ")[:1500]

    append_summary(row, summary_csv_local)
    if summary_csv_drive:
        append_summary(row, summary_csv_drive)

    if idx % 10 == 0 or idx == len(batch_paths):
        print(f"[{idx}/{len(batch_paths)}] OK={ok_count} FAIL={fail_count}")

print("\nDONE.")
print("Local master CSV:  ", master_csv_local)
print("Local summary CSV: ", summary_csv_local)
if master_csv_drive:
    print("Drive master CSV:  ", master_csv_drive)
if summary_csv_drive:
    print("Drive summary CSV: ", summary_csv_drive)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Batch size: 200
First 5: ['10128.jpg', '10180.jpg', '10183.jpg', '10208.jpg', '10214.jpg']
Last 5:  ['18717.jpg', '18731.jpg', '18817.jpg', '18869.jpg', '18929.jpg']
[10/200] OK=10 FAIL=0
[20/200] OK=20 FAIL=0
[30/200] OK=30 FAIL=0
[40/200] OK=39 FAIL=1
[50/200] OK=49 FAIL=1
[60/200] OK=59 FAIL=1
[70/200] OK=69 FAIL=1
[80/200] OK=79 FAIL=1
[90/200] OK=89 FAIL=1
[100/200] OK=98 FAIL=2
[110/200] OK=107 FAIL=3
[120/200] OK=117 FAIL=3
[130/200] OK=127 FAIL=3
[140/200] OK=136 FAIL=4
[150/200] OK=146 FAIL=4
[160/200] OK=156 FAIL=4
[170/200] OK=166 FAIL=4
[180/200] OK=176 FAIL=4
[190/200] OK=186 FAIL=4
[200/200] OK=196 FAIL=4

DONE.
Local master CSV:   /content/testcases_all_top200.csv
Local summary CSV:  /content/summary_top200.csv
Drive master CSV:   /content/drive/MyDrive/SS_PIPELINE_200_final1/testcases_csv/testcases_all_top200.csv
Drive summary CSV:  /content/drive/MyDrive/SS_PIPELINE_200_final1/summary_top200.csv
